# Structured output

Make an agent return validated, machine-readable data — a Pydantic object or dict — instead of prose you have to parse.

## Concept
Pass a schema to create_agent(..., response_format=...). The parsed result lands in the "structured_response" key of the final state.

- ProviderStrategy — native structured-output API (OpenAI, Anthropic, Gemini). Most reliable.
- ToolStrategy — tool-calling fallback that works on any model; supports Union types.

In [3]:
from attr import dataclass
from pydantic import SecretStr, BaseModel, Field
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address")
    phone: str = Field(description="The phone number")

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    response_format=ContactInfo       # auto-selects a strategy
)

result = agent.invoke({
    "messages": [{"role": "user",
        "content": "Extract: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


## Use ToolStrategy when you need validation constraints and enums:

In [7]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int = Field(description="Rating 1-5", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="Overall sentiment")
    key_points: list[str] = Field(description="Key points of the review")

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    response_format=ProductReview       # auto-selects a strategy
)

result = agent.invoke({
    "messages": [{"role": "user",
        "content": "Analyze: 'Great product, 5/5. Fast shipping, but pricey.'"}]
})
print(result["structured_response"].sentiment)  # positive

positive
